In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# ==========================================
# 1. INPUT EMBEDDINGS
# ==========================================

B = 2        # Batch size
T = 6        # Tokens
D = 8        # Embedding dimension
H = 4        # Attention heads

x = torch.randn(B, T, D)

print("Input:", x.shape)


# ==========================================
# 2. SIMPLE SELF-ATTENTION
# ==========================================

scores = x @ x.transpose(-2, -1)

weights = torch.softmax(scores, dim=-1)

context = weights @ x

print("\nSimple Self-Attention")
print("Scores:", scores.shape)
print("Weights:", weights.shape)
print("Context:", context.shape)


# ==========================================
# 3. TRAINABLE SELF-ATTENTION
# ==========================================

W_query = nn.Linear(D, D, bias=False)
W_key = nn.Linear(D, D, bias=False)
W_value = nn.Linear(D, D, bias=False)

Q = W_query(x)
K = W_key(x)
V = W_value(x)

scores = Q @ K.transpose(-2, -1)
scores = scores / (D ** 0.5)

weights = torch.softmax(scores, dim=-1)

context = weights @ V

print("\nTrainable Self-Attention")
print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)
print("Scores:", scores.shape)
print("Context:", context.shape)


# ==========================================
# 4. SCALED DOT-PRODUCT ATTENTION
#    USING PYTORCH BUILT-IN FUNCTION
# ==========================================

context = F.scaled_dot_product_attention(
    Q, K, V,
    dropout_p=0.0
)

print("\nScaled Dot-Product Attention")
print("Context:", context.shape)


# ==========================================
# 5. CAUSAL ATTENTION
# ==========================================

causal_context = F.scaled_dot_product_attention(
    Q, K, V,
    is_causal=True,
    dropout_p=0.0
)

print("\nCausal Attention")
print("Context:", causal_context.shape)


# ==========================================
# 6. MULTI-HEAD ATTENTION
# ==========================================

mha = nn.MultiheadAttention(
    embed_dim=D,
    num_heads=H,
    dropout=0.1,
    batch_first=True
)

# nn.MultiheadAttention sử dụng mask True
# để chặn các vị trí tương lai.
causal_mask = torch.triu(
    torch.ones(T, T, dtype=torch.bool),
    diagonal=1
)

mha.eval()

with torch.no_grad():
    output, attention_weights = mha(
        x, x, x,
        attn_mask=causal_mask,
        need_weights=True,
        average_attn_weights=False
    )

print("\nMulti-Head Attention")
print("Input:", x.shape)
print("Output:", output.shape)
print("Attention Weights:", attention_weights.shape)


# ==========================================
# 7. VERIFY CAUSAL MASK
# ==========================================

print("\nAttention Weights — Batch 1, Head 1")
print(attention_weights[0, 0])

print("\nFuture attention weights are zero:")
future_positions = torch.triu(
    torch.ones(T, T, dtype=torch.bool),
    diagonal=1
)

print(
    torch.all(
        attention_weights[0, 0][future_positions] == 0
    ).item()
)

Input: torch.Size([2, 6, 8])

Simple Self-Attention
Scores: torch.Size([2, 6, 6])
Weights: torch.Size([2, 6, 6])
Context: torch.Size([2, 6, 8])

Trainable Self-Attention
Q: torch.Size([2, 6, 8])
K: torch.Size([2, 6, 8])
V: torch.Size([2, 6, 8])
Scores: torch.Size([2, 6, 6])
Context: torch.Size([2, 6, 8])

Scaled Dot-Product Attention
Context: torch.Size([2, 6, 8])

Causal Attention
Context: torch.Size([2, 6, 8])

Multi-Head Attention
Input: torch.Size([2, 6, 8])
Output: torch.Size([2, 6, 8])
Attention Weights: torch.Size([2, 4, 6, 6])

Attention Weights — Batch 1, Head 1
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3458, 0.6542, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5397, 0.2119, 0.2484, 0.0000, 0.0000, 0.0000],
        [0.1211, 0.2860, 0.2043, 0.3886, 0.0000, 0.0000],
        [0.1864, 0.2606, 0.2666, 0.1856, 0.1007, 0.0000],
        [0.1012, 0.1949, 0.1736, 0.1615, 0.1298, 0.2389]])

Future attention weights are zero:
True


# Chapter 3 — PyTorch Attention Cheat Sheet

> Bảng tra cứu nhanh các hàm, tham số và tensor shapes từ Simple Self-Attention đến Multi-Head Attention. Ký hiệu: `B` = batch size, `T` = số token, `D` = embedding dimension, `H` = số heads, `Dh = D/H`.

## 1. Các hàm và lớp PyTorch

| Hàm / lớp | Cú pháp thường dùng | Tham số quan trọng | Mục đích |
|---|---|---|---|
| `torch.matmul` / `@` | `torch.matmul(a, b)` / `a @ b` | `a`, `b`: hai tensor tương thích phép nhân ma trận | Tính scores hoặc context |
| `torch.softmax` | `torch.softmax(scores, dim=-1)` | `input`: scores; `dim=-1`: chuẩn hóa theo chiều key | Chuyển scores thành attention weights |
| `nn.Linear` | `nn.Linear(in_features, out_features, bias=True)` | `in_features`, `out_features`, `bias` | Tạo phép chiếu Q/K/V có tham số học được |
| `F.scaled_dot_product_attention` | `F.scaled_dot_product_attention(Q, K, V, is_causal=True, dropout_p=0.0)` | `query`, `key`, `value`, `attn_mask`, `dropout_p`, `is_causal`, `scale` | Scaled dot-product attention tối ưu |
| `nn.MultiheadAttention` | `nn.MultiheadAttention(embed_dim=D, num_heads=H, batch_first=True)` | `embed_dim`, `num_heads`, `dropout`, `bias`, `batch_first` | Multi-head attention có sẵn |
| `nn.Dropout` | `nn.Dropout(p=0.1)` | `p`: xác suất loại bỏ | Regularization khi training |
| `torch.triu` | `torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)` | `input`, `diagonal=1` | Tạo mask tam giác phía trên |
| `Tensor.transpose` | `x.transpose(-2, -1)` | `dim0`, `dim1` | Đổi chỗ hai chiều, ví dụ K → Kᵀ |
| `Tensor.view` | `x.view(B, T, H, Dh)` | Kích thước mới; tensor phải có layout phù hợp | Tách chiều heads |
| `Tensor.reshape` | `x.reshape(B, T, D)` | Kích thước mới | Ghép lại chiều heads |
| `torch.cat` | `torch.cat(head_outputs, dim=-1)` | `tensors`, `dim` | Nối outputs từ các heads |

## 2. `nn.Linear` — Q, K, V

| Tham số | Bắt buộc? | Mặc định | Giải thích |
|---|---|---|---|
| `in_features` | Có | — | Chiều vector đầu vào |
| `out_features` | Có | — | Chiều vector sau phép chiếu |
| `bias` | Không | `True` | Có cộng bias hay không |

```python
W_q = nn.Linear(D, D, bias=False)
W_k = nn.Linear(D, D, bias=False)
W_v = nn.Linear(D, D, bias=False)
Q, K, V = W_q(x), W_k(x), W_v(x)
```

## 3. `F.scaled_dot_product_attention` — Scaled/Causal Attention

| Tham số | Bắt buộc? | Mặc định | Giải thích |
|---|---|---|---|
| `query` | Có | — | Tensor Q |
| `key` | Có | — | Tensor K |
| `value` | Có | — | Tensor V |
| `attn_mask` | Không | `None` | Mask tùy chỉnh; **boolean `True` = được phép attention** |
| `dropout_p` | Không | `0.0` | Xác suất dropout; hàm vẫn áp dụng nếu `dropout_p > 0`, kể cả khi model đang eval |
| `is_causal` | Không | `False` | Tự áp dụng causal mask khi `True` |
| `scale` | Không | `None` | `None` tương đương `1/sqrt(d_k)` |

```python
context = F.scaled_dot_product_attention(
    Q, K, V,
    is_causal=True,
    dropout_p=0.0,
)
```

**Lưu ý:** Tránh truyền đồng thời `attn_mask` và `is_causal=True` nếu muốn code tương thích rộng giữa các phiên bản PyTorch. Với dropout khi đánh giá: `dropout_p=(p if model.training else 0.0)`.

## 4. `nn.MultiheadAttention` — Khởi tạo

| Tham số | Bắt buộc? | Mặc định | Giải thích |
|---|---|---|---|
| `embed_dim` | Có | — | Chiều embedding đầu vào |
| `num_heads` | Có | — | Số heads; `embed_dim` phải chia hết cho `num_heads` |
| `dropout` | Không | `0.0` | Dropout trên attention weights |
| `bias` | Không | `True` | Bias trong phép chiếu |
| `batch_first` | Không | `False` | `True`: tensor `[B,T,D]`; `False`: `[T,B,D]` |
| `kdim`, `vdim` | Không | `None` | Chiều K/V nếu khác `embed_dim` |

```python
mha = nn.MultiheadAttention(
    embed_dim=8,
    num_heads=4,
    dropout=0.1,
    batch_first=True,
)
```

## 5. `mha(...)` — Gọi Multi-Head Attention

| Tham số | Bắt buộc? | Mặc định | Giải thích |
|---|---|---|---|
| `query`, `key`, `value` | Có | — | Với self-attention: `mha(x, x, x)` |
| `key_padding_mask` | Không | `None` | Che các token padding trong K |
| `need_weights` | Không | `True` | Có trả về attention weights không |
| `attn_mask` | Không | `None` | Mask theo cặp vị trí token; **boolean `True` = bị chặn** |
| `average_attn_weights` | Không | `True` | `False`: trả weights riêng từng head `[B,H,T,T]` |
| `is_causal` | Không | `False` | Hint causal; nên cung cấp causal `attn_mask` phù hợp để dùng ổn định |

```python
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
mha.eval()
with torch.no_grad():
    output, weights = mha(
        x, x, x,
        attn_mask=mask,
        need_weights=True,
        average_attn_weights=False,
    )
```

**Quan trọng:** Quy ước boolean mask **ngược nhau**:

| API | `True` trong boolean `attn_mask` |
|---|---|
| `F.scaled_dot_product_attention` | Cho phép attention |
| `nn.MultiheadAttention` | Chặn attention |

## 6. Tensor shapes qua từng bước

Giả sử `B=2`, `T=6`, `D=8`, `H=4`, `Dh=2`.

| Bước | Phép toán | Shape |
|---|---|---|
| Input | `x` | `[2,6,8]` |
| Q/K/V | `nn.Linear(8,8)(x)` | Mỗi tensor `[2,6,8]` |
| Tách heads | `.view(B,T,H,Dh).transpose(1,2)` | `[2,4,6,2]` |
| Scores mỗi head | `Q @ K.transpose(-2,-1)` | `[2,4,6,6]` |
| Weights | `softmax(scores / sqrt(Dh), dim=-1)` | `[2,4,6,6]` |
| Context mỗi head | `weights @ V` | `[2,4,6,2]` |
| Ghép heads | `.transpose(1,2).reshape(B,T,D)` | `[2,6,8]` |
| Output projection | `nn.Linear(D,D)` | `[2,6,8]` |
| Weights trả về từ `mha` | `average_attn_weights=False` | `[2,4,6,6]` |

## 7. Sơ đồ Q/K/V

```text
Input X [B,T,D]
   ├── Linear Wq ──> Q ──┐
   ├── Linear Wk ──> K ──┼──> Split heads [B,H,T,Dh]
   └── Linear Wv ──> V ──┘              │
                                       ▼
                             Scores = Q @ Kᵀ / √Dh
                                       │
                                Causal Mask
                                       │
                                    Softmax
                                       │
                               Weights @ V
                                       │
                                Merge heads
                                       │
                                Output Linear
                                       ▼
                                  [B,T,D]
```

## 8. Công thức cần nhớ

| Loại | Công thức | Điểm khác biệt |
|---|---|---|
| Simple Self-Attention | `softmax(X @ Xᵀ) @ X` | Không có Q/K/V học được |
| Trainable Self-Attention | `softmax(Q @ Kᵀ / √d_k) @ V` | Q/K/V qua các phép chiếu học được |
| Causal Attention | `softmax(mask(Q @ Kᵀ / √d_k)) @ V` | Mask vị trí tương lai bằng `-inf` trước softmax |
| Multi-Head Attention | `Concat(head_1,...,head_H) @ W_o` | Nhiều heads tính attention song song |

**Vì sao GPT không nhìn token tương lai?** GPT dự đoán token kế tiếp dựa trên những token đã xuất hiện. Causal mask buộc attention weight tới vị trí tương lai bằng 0, tránh rò rỉ đáp án trong huấn luyện và giữ cách xử lý phù hợp với lúc sinh văn bản.
